In [1]:
from Autograd import Tensor
from Activation import ReLU
from Loss import cross_entropy_loss
from Model import Linear,Sequential,Layernorm,Embedding,Flatten,Module
from Optimiser import SGD,Momentum,Adagrad,RMSProp,Adadelta,Adam
from Training import training
import random
import numpy as np
import pandas as pd

In [22]:
class Attention(Module):
    def __init__(self,value_dim,query_dim,embedding_dim,block_size):
        self.value_dim=value_dim
        self.query_dim=query_dim
        self.embedding_dim=embedding_dim
        self.mask=np.triu(np.ones((block_size,block_size)),k=1)
        self.Wv=Tensor(np.random.rand(embedding_dim,value_dim),requires_grad=True)
        self.Wk=Tensor(np.random.rand(embedding_dim,query_dim),requires_grad=True)
        self.Wq=Tensor(np.random.rand(embedding_dim,query_dim),requires_grad=True)
    def forward(self,x):
        v=x@self.Wv
        k=x@self.Wk
        q=x@self.Wq
        presoftmax=q@k.transpose()/self.query_dim**0.5
        presoftmax=presoftmax+Tensor(np.where(self.mask,-np.inf,0))
        softmax=presoftmax.softmax()
        return softmax@v

In [3]:
def data_read(prop):
    df=pd.read_csv("names.txt",header=None)
    data_list=df[0].tolist()
    random.shuffle(data_list)
    chars=sorted(set("".join(data_list)))
    chars.append(".")
    encodedict={}
    decodedict={}
    for id,char in enumerate(chars):
        encodedict[char]=id
        decodedict[id]=char
    length=len(data_list)
    n=int(length*prop)
    train_set=data_list[:n]
    test_set=data_list[n:]
    return train_set,test_set,encodedict,decodedict

def encode(name):
    return [encodedict["."]]+[encodedict[ch] for ch in name]+[encodedict["."]]

def decode(sequence):
    return "".join([decodedict[id] for id in sequence])

def tokeniser(data,block_size):
    tokens=encode(".".join(data)+".")
    X=[]
    Y=[]
    for i in range(len(tokens)-block_size):
        X.append(tokens[i:i+block_size])
        Y.append(tokens[i+1:i+1+block_size])
    return X,Y

In [10]:
block_size=10

train_set,test_set,encodedict,decodedict=data_read(0.1)
X,Y=tokeniser(train_set,block_size)
X,Y=Tensor(X),Tensor(Y)
no_characters=len(encodedict)

In [ ]:
embedding_dim=15

value_dim=10
query_dim=10

token_embedding=Embedding(no_characters,embedding_dim)
position_embedding=Embedding(block_size,embedding_dim)

attention=Attention(value_dim,query_dim,embedding_dim)

embeddings=token_embedding.forward(X)+position_embedding.forward(Tensor(np.arange(block_size)))
out=attention.forward(embeddings)

(22678, 10, 10)